In [1]:
import os
import sys
from pathlib import Path

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import nvitk.core as core
import nvitk.core.logger as logger
core.setup(globals())
log = logger.Logger()

import nvitk as nv
from nvitk.morphology.centerline_siphon import correct_siphon_centerlines

OUT_DIR = Path("/home/imarcoss/nvitk/notebooks/exploration/pesabrain-anatomy/checkpoints")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
tof = nv.imread(
    "/home/imarcoss/DATA/LabVF/PESA-Brain/WVI-BB/RESULTS/eicab_test/PESA15689521/TOF_resampled.nii.gz",
    backend='cpu', axes='XYZ'
)
eicab = nv.imread(
    "/home/imarcoss/DATA/LabVF/PESA-Brain/WVI-BB/RESULTS/eicab_test/PESA15689521/TOF_eICAB_CW.nii.gz",
    backend='cpu', axes='XYZ'
)
print("TOF:", tof, "\neICAB:", eicab)


TOF: Image(shape=(347, 401, 192), dtype=float32, backend=numpy, axes='XYZ', orientation='RAS', name='TOF_resampled.nii', modality=None, submodality=None, rescale_type='DV') 
eICAB: Image(shape=(347, 401, 192), dtype=float32, backend=numpy, axes='XYZ', orientation='RAS', name='TOF_eICAB_CW.nii', modality=None, submodality=None, rescale_type='DV')


In [ ]:
# tof = nv.imread(
#     "/home/imarcoss/NetVolumes/LAB_MCC/LabVF/PESA-Brain/RESULTS/res_QVTPy/PESA15689521/eicab/TOF_resampled.nii.gz",
#     backend='cpu', axes='XYZ'
# )
# eicab = nv.imread(
#     "/home/imarcoss/NetVolumes/LAB_MCC/LabVF/PESA-Brain/RESULTS/res_QVTPy/PESA15689521/eicab/TOF_eICAB_CW.nii.gz",
#     backend='cpu', axes='XYZ'
# )
# print("TOF:", tof, "\neICAB:", eicab)


In [7]:
res = correct_siphon_centerlines(
    tof, eicab,
    correction_ids=(1, 2),
    out_dir=OUT_DIR,
    save_qc=True,
)

16:11:21 | INFO     |   ▸ correct_siphon_centerlines: shape=(347, 401, 192) correction_ids=(1, 2)
16:11:21 | INFO     |   ▸ labels present: [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 16, 17, 18] | siphon-corrected: [1, 2] | default: [3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 16, 17, 18]
16:11:21 | INFO     |   ▸ === Seed centerlines from vessel_mask ===
16:11:25 | INFO     |   ▸ seed centerlines done in 3.40s (16 labels)
16:11:25 | INFO     |   ▸ === Default centerlines for non-ICA labels ===
16:11:27 | INFO     |   ▸ === ICA Otsu + repair + siphon centerlines ===
16:11:27 | INFO     |   ▸ --- LICA (id=1) ---
16:11:27 | INFO     |   ▸ [LICA] Otsu+erode in 0.17s: thr=58322.813907216 vox_pre=7190 → vox_post=2598 (erode_iters=2) bbox=(136, 160, 221, 400, 18, 74)
16:11:29 | INFO     |   ▸ [LICA] eroded (iters=2): voxels=2598 β₀=1 χ=0.0 β₁=1 skel_cycles=1 suspect=True
16:11:29 | INFO     |   ▸ [LICA] eroded mask still suspect → running 3D donut cut
16:11:29 | INFO     |   ▸ [LICA] iter 0: cyc


ICA      vox_o   vox_e   vox_r   β₁ o→e→r    cyc o→e→r  CL_pts                           action
--------------------------------------------------------------------------------------------------------------
LICA      7190    2598    2598      2→1→1        7→1→1      85                          partial
RICA      6933    2522    2522      1→2→2        8→1→1      82                          partial


In [5]:
res.get('details')

{1: {'label': 1,
  'label_name': 'LICA',
  'n_skel': 77,
  'n_skel_pruned': 77,
  'n_bridge': 0,
  'n_pts': 77,
  'base': [155, 235, 39],
  'tip': [152, 245, 70],
  'cycles': [],
  'bridge_voxels': [],
  'warning': None,
  'prep': {'otsu_info': {'label': 'LICA',
    'otsu_thresh': 58736.22604381132,
    'bbox': (144, 169, 228, 405, 35, 76),
    'n_voxels': 2127,
    'n_voxels_pre_erode': 6201,
    'erode_iters': 2},
   'otsu_mask': array([[[False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
           ...,
           [False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False]],
   
          [[False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
      